# 01 - Data Quality Assessment

In this notebook we **profile the raw datasets** and document the data quality issues that will be remediated in `02_data_cleaning.ipynb`.

The goal is **not** to fix anything yet, only to inventory the problems so we can decide on the right remediation strategy.

**Framework used (per dataset):**

| Problem | Impact | Solution |
|---------|--------|----------|
| ...     | ...    | ...      |

We look for:
- Missing values (nulls)
- Duplicate rows
- Wrong / inconsistent data types
- Inconsistent categorical labels
- Outliers and impossible values
- Date format inconsistencies

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

RAW = ROOT / 'data' / 'raw'
customers = pd.read_csv(RAW / 'customers.csv')
products  = pd.read_csv(RAW / 'products.csv')
orders    = pd.read_csv(RAW / 'orders.csv')
items     = pd.read_csv(RAW / 'order_items.csv')
returns   = pd.read_csv(RAW / 'returns.csv')

for name, df in [('customers', customers), ('products', products),
                 ('orders', orders), ('order_items', items),
                 ('returns', returns)]:
    print(f'{name:>11}: {df.shape[0]:>6,} rows x {df.shape[1]} cols')

## Quality profiling helpers

We will reuse these helpers for every table.

In [ ]:
def null_report(df: pd.DataFrame) -> pd.DataFrame:
    n = df.isna().sum()
    pct = (n / len(df) * 100).round(2)
    return pd.DataFrame({'nulls': n, 'null_pct': pct}).query('nulls > 0').sort_values('nulls', ascending=False)

def dup_report(df: pd.DataFrame) -> int:
    return int(df.duplicated().sum())

def dtype_report(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({'dtype': df.dtypes.astype(str), 'sample': [df[c].dropna().head(1).tolist() for c in df.columns]})

## 1. customers.csv

In [ ]:
print('Shape:', customers.shape)
print('Duplicates:', dup_report(customers))
print()
print('Nulls:')
display(null_report(customers))
print('\nDtypes:')
display(dtype_report(customers))

In [ ]:
print('Distinct region values (note casing & whitespace):')
print(customers['region'].value_counts(dropna=False).to_string())
print('\nDistinct segment values:')
print(customers['segment'].value_counts(dropna=False).to_string())

In [ ]:
customers['signup_date'] = pd.to_datetime(customers['signup_date'], errors='coerce')
future = customers[customers['signup_date'] > pd.Timestamp('2024-12-31')]
print(f'Signup dates beyond 2024-12-31: {len(future):,}')
display(future.head(3))

### customers.csv - issues summary

| Problem | Impact | Solution |
|---------|--------|----------|
| Duplicate rows (exact) | Inflated customer counts, double-counted revenue | Drop exact duplicates, keep first |
| Null emails / phones | Cannot contact customer; segmentation gaps | Leave as NaN (kept for analysis, treated as 'Unknown' in segments) |
| Null region | Geography segmentation fails | Impute with `'Unknown'` bucket |
| Mixed case in `first_name`/`last_name` | Inconsistent reports | Title-case and trim |
| Inconsistent region labels (`North`/`NORTH`/`north `) | Region rollups are wrong | Map to canonical labels |
| Future signup dates | Impossible values | Set to NaT (treated as missing) |

## 2. products.csv

In [ ]:
print('Shape:', products.shape)
print('Duplicates:', dup_report(products))
print('\nNulls:')
display(null_report(products))
print('\nDtypes:')
display(dtype_report(products))

In [ ]:
print('Distinct categories:')
print(products['category'].value_counts(dropna=False).to_string())
print('\nDistinct sub-categories:')
print(products['sub_category'].value_counts(dropna=False).to_string())

In [ ]:
print(f'Negative unit_cost: {(products["unit_cost"] < 0).sum():,}')
print(f'Negative unit_price: {(products["unit_price"] < 0).sum():,}')
print(f'Negative stock_units: {(products["stock_units"] < 0).sum():,}')
print('\nUnit price distribution:')
print(products['unit_price'].describe().round(2))

### products.csv - issues summary

| Problem | Impact | Solution |
|---------|--------|----------|
| Duplicate products | Same SKU counted twice in catalogs | Drop exact duplicates |
| Null categories | Category rollups fail | Impute with `'Uncategorized'` |
| Inconsistent category spelling (`ELECTRONICS`, `electronic`, `Elec.`) | Categories split | Map to canonical labels |
| Negative unit_cost / unit_price | Distorts revenue & profit | Set to NaN then impute by category median |
| Extreme unit_price outliers (x50) | Skews pricing analysis | Cap at 99th percentile |
| Whitespace in product_name | Bad lookups & joins | Trim & collapse whitespace |

## 3. orders.csv

In [ ]:
print('Shape:', orders.shape)
print('Duplicates:', dup_report(orders))
print('\nNulls:')
display(null_report(orders))

In [ ]:
orders['order_date_parsed'] = pd.to_datetime(orders['order_date'], errors='coerce')
print('Raw date strings that could NOT be parsed:')
print(orders[orders['order_date_parsed'].isna()][['order_date']].head(10).to_string())

print(f'\nFuture order dates (after 2024-12-31): {((orders["order_date_parsed"] > pd.Timestamp("2024-12-31")) & orders["order_date_parsed"].notna()).sum():,}')

In [ ]:
print('Distinct payment_method values:')
print(orders['payment_method'].value_counts(dropna=False).to_string())
print('\nDistinct status values:')
print(orders['status'].value_counts(dropna=False).to_string())
print('\nDistinct region values:')
print(orders['region'].value_counts(dropna=False).to_string())

### orders.csv - issues summary

| Problem | Impact | Solution |
|---------|--------|----------|
| Duplicate orders | Inflated order count & revenue | Drop exact duplicates |
| Null `customer_id` | Order cannot be attributed | Keep order but flag as orphan |
| Mixed date formats | Time-series aggregations fail | Parse with multi-format strategy |
| Inconsistent region (`north `, ` SOUTH`) | Geography rollups split | Normalize to canonical labels |
| Inconsistent `payment_method` (`CC`, `credit card`, `CREDIT CARD`) | Payment mix analysis wrong | Map to canonical labels |
| Inconsistent `status` (`complete`, `COMPLETE`) | Fulfillment metrics wrong | Map to canonical labels |
| Future order dates | Impossible values | Set to NaT |

## 4. order_items.csv

In [ ]:
print('Shape:', items.shape)
print('Duplicates:', dup_report(items))
print('\nNulls:')
display(null_report(items))

In [ ]:
print(f'Negative quantity: {(items["quantity"] < 0).sum():,}')
print(f'Zero unit_price: {(items["unit_price"] == 0).sum():,}')
print(f'Negative unit_price: {(items["unit_price"] < 0).sum():,}')
print(f'Discounts > 1: {(items["discount"] > 1).sum():,}')
print(f'Negative discounts: {(items["discount"] < 0).sum():,}')

print('\nQuantity distribution (showing tail):')
print(items['quantity'].describe().round(2))

### order_items.csv - issues summary

| Problem | Impact | Solution |
|---------|--------|----------|
| Duplicate line items | Inflated revenue | Drop exact duplicates |
| Null `order_id` / `product_id` | Cannot join | Keep, downstream join will produce NaN |
| Negative quantities | Reversed lines, wrong revenue | Set to NaN, default to 1 |
| Outlier quantities (100-5000) | Skew averages | Cap to 99th percentile |
| Zero / negative `unit_price` | Zero / negative revenue | Replace with NaN, impute with median |
| Discounts outside [0,1] | Inflated discount | Clip to [0, 0.9] |

## 5. returns.csv

In [ ]:
print('Shape:', returns.shape)
print('Duplicates:', dup_report(returns))
print('\nNulls:')
display(null_report(returns))
print('\nDistinct reasons:')
print(returns['reason'].value_counts(dropna=False).to_string())
print(f'\nNegative refunds: {(returns["refund_amount"] < 0).sum():,}')

### returns.csv - issues summary

| Problem | Impact | Solution |
|---------|--------|----------|
| Duplicate returns | Double-counted refunds | Drop exact duplicates |
| Null reason | Categorical analysis incomplete | Impute with `'Unknown'` |
| Inconsistent reason labels (`dmg`, `NAD`, `WrongItem`) | Reason rollups split | Map to canonical labels |
| Negative refund amounts | Wrong refund totals | Take absolute value |
| Future return dates | Impossible values | Set to NaT |

## Summary of the data-quality findings

Across the five raw files we identified the following issue categories:

1. **Duplicates** in every file (15 to 250 rows).
2. **Nulls** in several columns (emails, phones, regions, categories, reasons).
3. **Inconsistent categorical labels** for `region`, `category`, `payment_method`, `status`, `reason`.
4. **Impossible numeric values**: negative prices, costs, quantities, discounts.
5. **Outliers**: extreme unit prices (x50), extreme quantities (100-5000), extreme refunds.
6. **Mixed date formats** in `orders.csv` plus future dates in `customers`, `orders`, `returns`.

These issues are remediated in `02_data_cleaning.ipynb` using the helpers in `src/cleaning.py`.